# 02 — Evaluate PERPHECT Model

Load a trained model and the held-out test set (precomputed in `01_prepare_test_set.ipynb`),
then display a full metrics suite:
- Classification report (precision, recall, F1)
- MCC, sensitivity, specificity
- Confusion matrix
- ROC-AUC curve
- Precision-recall curve
- Prediction distribution histogram
- Per-source metrics
- Threshold analysis
- Multi-run comparison

**No database connection required** — all data is loaded from disk.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
)

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

## 1. Configuration

In [ ]:
# Where trained models live (this directory or a subdirectory)
OUTPUTS_DIR = Path("/results")

# Held-out test set from 01_prepare_test_set.ipynb
TEST_DATA = Path.cwd() / "test_data"

# Prediction threshold
THRESHOLD = 0.5

## 2. Load Test Data

In [ ]:
npz_path = TEST_DATA / "test_set.npz"
csv_path = TEST_DATA / "test_set.csv"

if not npz_path.exists():
    raise FileNotFoundError(
        f"No test data found at {npz_path}.\n"
        "Re-run 01_prepare_test_set.ipynb first."
    )

data = np.load(npz_path, allow_pickle=True)
bacteria_arr = data["bacteria"]
phage_arr = data["phage"]
labels = data["labels"]
sources = data["sources"]

# Load string IDs if available
test_csv = pd.read_csv(csv_path) if csv_path.exists() else None

print(f"Loaded {len(labels)} test pairs from {npz_path}")
print(f"Bacteria: {bacteria_arr.shape}, Phage: {phage_arr.shape}")
print(f"\nLabel distribution: {dict(zip(*np.unique(labels, return_counts=True)))}")
print(f"Source distribution: {dict(zip(*np.unique(sources, return_counts=True)))}")
if test_csv is not None:
    print(f"\nTest CSV columns: {list(test_csv.columns)}")
    print(test_csv.head())

## 3. Discover Available Runs

In [ ]:
def find_runs(base_dir):
    """Find all training runs and extract their summaries."""
    runs = []
    # Standard splits: base_dir/run_name/summary.json
    for summary in sorted(base_dir.glob("*/summary.json")):
        run_dir = summary.parent
        runs.append({"dir": run_dir, "summary_file": summary, "type": "standard"})
    # Cross-validation: base_dir/run_name/cv_summary.json
    for cv_summary in sorted(base_dir.glob("*/cv_summary.json")):
        run_dir = cv_summary.parent
        runs.append({"dir": run_dir, "summary_file": cv_summary, "type": "cv"})
    return runs

available_runs = find_runs(OUTPUTS_DIR)

if not available_runs:
    print(f"No training runs found under {OUTPUTS_DIR}")
    print("Set OUTPUTS_DIR to the directory containing run_<timestamp>/ subdirectories.")
else:
    print(f"Found {len(available_runs)} run(s):\n")
    for i, run in enumerate(available_runs):
        run_name = run["dir"].name
        run_type = run["type"]
        with open(run["summary_file"]) as f:
            import json
            summary = json.load(f)
        auc_val = summary.get("best_val_auc", summary.get("mean_val_auc", "?"))
        print(f"  [{i}] {run_name}  (type={run_type}, val_auc={auc_val})")

## 4. Load Model

Models trained with focal loss need `custom_objects` to load.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))

from train import focal_loss

def load_model(run_dir):
    """Load the best model from a run directory."""
    import keras
    model_path = run_dir / "model_best.keras"
    if not model_path.exists():
        model_path = run_dir / "model_final.keras"
    if not model_path.exists():
        raise FileNotFoundError(f"No .keras model found in {run_dir}")
    model = keras.models.load_model(
        model_path,
        custom_objects={"focal_loss": focal_loss(0.25, 2.0)},
    )
    return model, model_path


# Load the first available run
if available_runs:
    selected_run = available_runs[0]
    model, model_path = load_model(selected_run["dir"])
    print(f"Loaded model from: {model_path}")
    model.summary()
    print(f"\nParameters: {model.count_params():,}")
else:
    print("No run to load.")

## 5. Generate Predictions

In [ ]:
predictions = model.predict([bacteria_arr, phage_arr], verbose=1).flatten()
pred_labels = (predictions > THRESHOLD).astype(int)

# Core metrics
mcc = matthews_corrcoef(labels, pred_labels)
tn, fp, fn, tp = confusion_matrix(labels, pred_labels).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Threshold: {THRESHOLD}")
print(f"\nPredictions: {len(predictions)}")
print(f"  True Positives:  {tp}")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"\nMCC:          {mcc:.4f}")
print(f"Sensitivity:  {sensitivity:.4f}")
print(f"Specificity:  {specificity:.4f}")

## 6. Classification Report

In [ ]:
print(classification_report(
    labels, pred_labels,
    target_names=["Negative", "Positive"],
))

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(labels, pred_labels)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title("Confusion Matrix")
plt.colorbar(im, ax=ax)
tick_marks = np.arange(2)
ax.set_xticks(tick_marks)
ax.set_xticklabels(["Negative", "Positive"])
ax.set_yticks(tick_marks)
ax.set_yticklabels(["Negative", "Positive"])
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

## 8. ROC-AUC Curve

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(labels, predictions)
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="darkorange", lw=2,
        label=f"ROC curve (AUC = {roc_auc_val:.3f})")
ax.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
print(f"ROC-AUC: {roc_auc_val:.4f}")

## 9. Precision-Recall Curve

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(labels, predictions)
avg_precision = average_precision_score(labels, predictions)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall, precision, color="blue", lw=2,
        label=f"PR curve (AP = {avg_precision:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()
print(f"Average Precision: {avg_precision:.4f}")

## 10. Prediction Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

pos_preds = predictions[labels == 1]
neg_preds = predictions[labels == 0]

ax.hist(neg_preds, bins=50, alpha=0.6, label="True Negative", color="steelblue")
ax.hist(pos_preds, bins=50, alpha=0.6, label="True Positive", color="coral")
ax.axvline(x=THRESHOLD, color="black", linestyle="--",
           label=f"Threshold ({THRESHOLD})")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Count")
ax.set_title("Prediction Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Threshold Analysis

Explore how MCC, sensitivity, and specificity change with the decision threshold.

In [ ]:
thresholds_range = np.arange(0.05, 0.95, 0.05)
mccs = []
sens = []
specs = []

for t in thresholds_range:
    pl = (predictions > t).astype(int)
    mccs.append(matthews_corrcoef(labels, pl))
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(labels, pl).ravel()
    sens.append(tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0)
    specs.append(tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds_range, mccs, "o-", label="MCC", color="green", lw=2)
ax.plot(thresholds_range, sens, "s-", label="Sensitivity", color="coral", lw=2)
ax.plot(thresholds_range, specs, "^-", label="Specificity", color="steelblue", lw=2)
ax.axvline(x=THRESHOLD, color="gray", linestyle="--", alpha=0.5, label=f"Current threshold ({THRESHOLD})")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("MCC, Sensitivity, Specificity vs Threshold")
ax.set_ylim([0, 1.05])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = np.argmax(mccs)
print(f"Best MCC: {mccs[best_idx]:.4f} at threshold {thresholds_range[best_idx]:.2f}")
print(f"  At this threshold: sensitivity={sens[best_idx]:.4f}, specificity={specs[best_idx]:.4f}")

## 12. Per-Source Metrics

In [ ]:
unique_sources = sorted(set(sources))

source_results = []
for source in unique_sources:
    mask = sources == source
    if mask.sum() == 0:
        continue
    src_labels = labels[mask]
    src_preds = pred_labels[mask]
    src_probs = predictions[mask]

    print(f"\n{'='*50}")
    print(f"Source: {source} ({mask.sum()} pairs)")
    print(f"{'='*50}")
    print(classification_report(
        src_labels, src_preds,
        target_names=["Negative", "Positive"],
        zero_division=0,
    ))

    mcc_s = matthews_corrcoef(src_labels, src_preds)
    print(f"  MCC: {mcc_s:.4f}")
    source_results.append({"source": source, "n": mask.sum(), "mcc": mcc_s})

    if len(set(src_labels)) > 1:
        fpr_s, tpr_s, _ = roc_curve(src_labels, src_probs)
        print(f"  ROC-AUC: {auc(fpr_s, tpr_s):.4f}")

## 13. Explore Folds (Cross-Validation)

If the run used `--cross-validate`, each fold has its own `summary.json` and pair CSVs.

In [ ]:
def explore_folds(run_dir):
    """Load and display per-fold results from a CV run."""
    folds = sorted(run_dir.glob("fold_*"))
    if not folds:
        print(f"No fold_* directories in {run_dir}")
        return

    fold_data = []
    for fold_dir in folds:
        summary_file = fold_dir / "summary.json"
        if not summary_file.exists():
            continue
        with open(summary_file) as f:
            import json
            summary = json.load(f)

        fold_data.append({
            "fold": fold_dir.name,
            "train_size": summary.get("train_size", "?"),
            "val_size": summary.get("val_size", "?"),
            "best_val_auc": summary.get("best_val_auc", "?"),
            "test_mcc": summary.get("test_mcc", "?"),
            "test_sensitivity": summary.get("test_sensitivity", "?"),
            "test_specificity": summary.get("test_specificity", "?"),
            "epochs_trained": summary.get("epochs_trained", "?"),
        })

        # Show per-fold pair splits
        for split_name in ["pairs_train.csv", "pairs_val.csv", "pairs_test.csv"]:
            split_path = fold_dir / split_name
            if split_path.exists():
                df = pd.read_csv(split_path)
                source_counts = df["source"].value_counts().to_dict() if "source" in df.columns else {}
                fold_data[-1][f"{split_name.replace('.csv', '')}_sources"] = source_counts

    df = pd.DataFrame(fold_data)
    print(f"Cross-validation results ({len(folds)} folds):\n")
    print(df[["fold", "train_size", "val_size", "best_val_auc",
             "test_mcc", "test_sensitivity", "test_specificity", "epochs_trained"]].to_string(index=False))

    # Summary stats
    numeric_cols = ["best_val_auc", "test_mcc", "test_sensitivity", "test_specificity"]
    for col in numeric_cols:
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        if len(vals) > 0:
            print(f"\n{col}: {vals.mean():.4f} +/- {vals.std():.4f}")

    return df


if available_runs:
    selected = available_runs[0]
    print(f"Exploring folds for: {selected['dir'].name}\n")
    fold_df = explore_folds(selected["dir"])

## 14. Compare Multiple Runs

Side-by-side comparison of different training runs.

In [ ]:
def compare_runs(runs_list, bacteria_arr, phage_arr, labels, sources, threshold=0.5):
    """Compare multiple runs on the same test set."""
    results = []
    for run in runs_list:
        run_dir = run["dir"]
        run_name = run_dir.name
        try:
            model, model_path = load_model(run_dir)
            preds = model.predict([bacteria_arr, phage_arr], verbose=0).flatten()
            pred_l = (preds > threshold).astype(int)

            mcc = matthews_corrcoef(labels, pred_l)
            tn, fp, fn, tp = confusion_matrix(labels, pred_l).ravel()
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            fpr_r, tpr_r, _ = roc_curve(labels, preds)
            roc_auc_r = auc(fpr_r, tpr_r)

            results.append({
                "run": run_name,
                "type": run["type"],
                "roc_auc": roc_auc_r,
                "mcc": mcc,
                "sensitivity": sens,
                "specificity": spec,
                "n_params": model.count_params(),
            })
            print(f"Loaded: {run_name}")
        except Exception as e:
            print(f"Failed to load {run_name}: {e}")

    return pd.DataFrame(results)


if len(available_runs) >= 2:
    comparison = compare_runs(
        available_runs[:5], bacteria_arr, phage_arr, labels, sources, THRESHOLD
    )
    print("\n")
    print(comparison.to_string(index=False))
else:
    print("Need at least 2 runs to compare. Showing single-run summary.")
    if available_runs:
        comparison = compare_runs(
            available_runs[:1], bacteria_arr, phage_arr, labels, sources, THRESHOLD
        )
        print(comparison.to_string(index=False))

## 15. Save Evaluation Results

In [ ]:
results_df = pd.DataFrame({
    "true_label": labels.astype(int),
    "predicted_prob": predictions,
    "predicted_label": pred_labels,
    "source": sources,
})

# Add string IDs if available
if test_csv is not None:
    if "Phage_ID" in test_csv.columns:
        results_df["phage_id"] = test_csv["Phage_ID"].values
    if "Host_ID" in test_csv.columns:
        results_df["host_id"] = test_csv["Host_ID"].values

results_path = TEST_DATA / "evaluation_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved evaluation results to: {results_path}")
print(f"Columns: {list(results_df.columns)}")
print(results_df.head(10))

## Summary

| Metric | Value |
|--------|-------|
| ROC-AUC | See section 8 |
| Average Precision | See section 9 |
| MCC | See section 5 |
| Sensitivity | See section 5 |
| Specificity | See section 5 |
| F1 (Positive) | See section 6 |
| Best threshold | See section 11 |
| Per-source breakdown | See section 12 |

**Interpretation guide:**
- **ROC-AUC > 0.9**: Excellent discrimination
- **ROC-AUC 0.7–0.9**: Good discrimination
- **ROC-AUC < 0.7**: Model needs improvement
- **MCC**: Range [-1, 1]. 0 = random, 1 = perfect, -1 = inverse
- High sensitivity = catches most true positives (low false negatives)
- High specificity = few false positives (reliable positive predictions)